# RPF Clean Walk-Forward Implementation Notebook

This notebook is the clean operational path for RPF walk-forward work.

It intentionally excludes the previous experimental branches and mixed artifact logic. A run is valid here only if it follows the current corrected contract and writes explicit `effective_*` router artifacts.

Heavy stages are included as executable notebook cells. They are controlled by explicit run flags in the setup cell so `Run All` cannot accidentally launch long jobs unless you enable them.


## 0. Clean Contract

Active path only:

```text
readiness
-> rank_signal_router
-> effective selected output inspection
-> HMM market-regime diagnostic
-> market recursive CUSUM
-> HMM-transition CUSUM
-> prequential regime-aware gate simulation
-> validation and decision checklist
```

Implementation contract:

- HMM detects persistent market context, not UP/DOWN labels.
- CUSUM detects change risk, not trade direction.
- CatBoostRanker remains the row scorer.
- The regime/change layer only allows, suppresses, or reduces signal budget.
- Current prediction labels and current prediction outcomes are diagnostics only.
- Shadow candidate output is for regime research, not strategy performance.
- Effective selected output is the strategy-performance stream.
- Do not use old regime/suppression artifacts that do not declare detector names and prediction source.
- Heavy stages are real notebook executions controlled by `RUN_READINESS`, `RUN_ROUTER`, `RUN_REGIME_DIAGNOSTIC`, `RUN_VALIDATION`, and `RUN_WORKTREE_CHECK`.


## 1. Setup

Run this first. It finds the project root and defines shared constants.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import polars as pl


def find_project_root(start: Path | None = None) -> Path:
    for env_name in ("RISKYIELDMM_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name, "").strip()
        if value:
            candidate = Path(value).expanduser().resolve()
            if (candidate / "regression_feature_engineering").exists() and (
                candidate / "test_output"
            ).exists():
                return candidate
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "regression_feature_engineering").exists() and (
            candidate / "test_output"
        ).exists():
            return candidate
    raise RuntimeError("Could not find RiskYieldMM project root")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PY = os.environ.get("PY", "/media/przem/linux_data/conda/envs/ml_env/bin/python")

# One switch for a full notebook-driven workflow. This notebook is intended
# to execute the whole clean workflow from VS Code/Jupyter. Set this to False
# only when you want a dry-run/control-panel pass.
RUN_FULL_WORKFLOW = True

# Per-stage switches. Keep these linked for full execution, or override a
# single stage to continue/re-run only part of the workflow.
RUN_READINESS = RUN_FULL_WORKFLOW
RUN_ROUTER = RUN_FULL_WORKFLOW
RUN_REGIME_DIAGNOSTIC = RUN_FULL_WORKFLOW
RUN_VALIDATION = RUN_FULL_WORKFLOW
RUN_WORKTREE_CHECK = RUN_FULL_WORKFLOW

STREAM_SUBPROCESS_OUTPUT = True
NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_notebook")
LOG_ROOT = PROJECT_ROOT / "test_output" / "rpf_walkforward_notebook_logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

ASSET = "BTCUSDT"
ROOT = "8h/B"
READINESS_TARGET = "target_reg_direction_extreme_up_share_hvol_v2"
CONFIG_PATH = "regression_feature_engineering/configs/rpf_clean_walkforward_v1.json"

ROUTER_CANDIDATE_NAMES = "up_none_v1,up_rocket_64_v1,down_none_v1,down_rocket_16_diag_v1"
HMM_STATE_COUNT = 3
HMM_SELECTION_MODE = "bic_v1"
HMM_STATE_COUNT_CHOICES = "2,3,4"
HMM_FILTER_MODE = "causal_forward_v1"
HMM_PCA_COMPONENTS = 5
HMM_MIN_HISTORY_WINDOWS = 80
HMM_LOOKBACK_WINDOWS = 240
HMM_REFIT_INTERVAL_WINDOWS = 20
MARKET_REGIME_MAX_FEATURES_PER_FAMILY = 16
MARKET_REGIME_LOOKBACK_BATCHES = 20
REGIME_CHANGE_DETECTORS = (
    "market_context_zshift_v1,"
    "market_context_recursive_cusum_v1,"
    "hmm_transition_cusum_v1"
)
MARKET_ZSHIFT_THRESHOLD = 3.0
RECURSIVE_CUSUM_K = 0.50
RECURSIVE_CUSUM_H = 5.0
CUSUM_STANDARDIZATION = "rolling_robust_z_v1"
CUSUM_TARGET_EVENT_RATE_MIN = 0.05
CUSUM_TARGET_EVENT_RATE_MAX = 0.25
CHANGE_LOOKBACK_WINDOWS = 60


print("PROJECT_ROOT:", PROJECT_ROOT)
print("PY:", PY)
print("NOTEBOOK_RUN_ID:", NOTEBOOK_RUN_ID)
print("LOG_ROOT:", LOG_ROOT)
print("RUN_FULL_WORKFLOW:", RUN_FULL_WORKFLOW)
print("RUN_READINESS:", RUN_READINESS)
print("RUN_ROUTER:", RUN_ROUTER)
print("RUN_REGIME_DIAGNOSTIC:", RUN_REGIME_DIAGNOSTIC)
print("RUN_VALIDATION:", RUN_VALIDATION)
print("RUN_WORKTREE_CHECK:", RUN_WORKTREE_CHECK)
print("ROUTER_CANDIDATE_NAMES:", ROUTER_CANDIDATE_NAMES)
print("REGIME_CHANGE_DETECTORS:", REGIME_CHANGE_DETECTORS)
print("HMM_STATE_COUNT:", HMM_STATE_COUNT)
print("HMM_SELECTION_MODE:", HMM_SELECTION_MODE)
print("HMM_STATE_COUNT_CHOICES:", HMM_STATE_COUNT_CHOICES)
print("HMM_FILTER_MODE:", HMM_FILTER_MODE)
print("HMM_LOOKBACK_WINDOWS:", HMM_LOOKBACK_WINDOWS)
print("CHANGE_LOOKBACK_WINDOWS:", CHANGE_LOOKBACK_WINDOWS)
print("RECURSIVE_CUSUM_K/H:", RECURSIVE_CUSUM_K, RECURSIVE_CUSUM_H)
print("CUSUM_STANDARDIZATION:", CUSUM_STANDARDIZATION)
print("CUSUM_TARGET_EVENT_RATE:", CUSUM_TARGET_EVENT_RATE_MIN, CUSUM_TARGET_EVENT_RATE_MAX)


## 1A. HMM/CUSUM Dependency Check

Check that the active Python environment can run the HMM stage. Recursive CUSUM is implemented in the repo and does not need an external dependency.


In [ ]:
def python_module_available(python_exe: str, module_name: str) -> tuple[bool, str]:
    script = (
        "import importlib, sys; "
        f"m=importlib.import_module({module_name!r}); "
        "print(getattr(m, '__version__', 'available'))"
    )
    result = subprocess.run(
        [python_exe, "-c", script],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode == 0:
        return True, result.stdout.strip()
    return False, result.stderr.strip() or result.stdout.strip()


HMM_AVAILABLE, HMM_VERSION_OR_ERROR = python_module_available(PY, "hmmlearn")
print("HMM_AVAILABLE:", HMM_AVAILABLE)
print("hmmlearn:", HMM_VERSION_OR_ERROR)
if not HMM_AVAILABLE:
    print("Install command:")
    print(f"{PY} -m pip install hmmlearn")

## 1B. Workflow Toggles And Runtime Settings

The notebook contains the full heavy workflow. By default, `RUN_FULL_WORKFLOW = True`, so `Run All` launches the heavy stages. Set it to `False` for a dry-run/control-panel pass.

Each executed stage streams output into the notebook and writes a log under:

```text
test_output/rpf_walkforward_notebook_logs/
```


## 2. User Inputs And Artifact Discovery

Panel paths are auto-discovered from the latest local feature-panel artifacts unless you export explicit paths before launching Jupyter.

This cell also prints the HMM/CUSUM/router configuration that will be used by the heavy stages.


In [ ]:
def latest_path(pattern: str) -> str:
    matches = sorted(
        PROJECT_ROOT.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True
    )
    return str(matches[0]) if matches else ""


def env_or_latest(name: str, pattern: str) -> str:
    value = os.environ.get(name, "").strip()
    if value and not (value.startswith("<") and value.endswith(">")):
        return value
    return latest_path(pattern)


UP_PANEL = env_or_latest(
    "UP_PANEL",
    "test_output/rpf_feature_panels/*up_ge_2x_down*/selected_panel_160.json",
)
UP_SEQ_PANEL = env_or_latest(
    "UP_SEQ_PANEL",
    "test_output/rpf_cnn_feature_diagnostics/*up_ge_2x_down*/selected_cnn_panel_160.json",
)
DOWN_PANEL = env_or_latest(
    "DOWN_PANEL",
    "test_output/rpf_feature_panels/*down_ge_2x_up*/selected_panel_160.json",
)
DOWN_SEQ_PANEL = env_or_latest(
    "DOWN_SEQ_PANEL",
    "test_output/rpf_cnn_feature_diagnostics/*down_ge_2x_up*/selected_cnn_panel_160.json",
)

READINESS_RUN = (
    Path(os.environ["READINESS_RUN"]) if "READINESS_RUN" in os.environ else None
)
ROUTER_RUN = Path(os.environ["ROUTER_RUN"]) if "ROUTER_RUN" in os.environ else None
REGIME_RUN = Path(os.environ["REGIME_RUN"]) if "REGIME_RUN" in os.environ else None

panel_paths = {
    "UP_PANEL": UP_PANEL,
    "UP_SEQ_PANEL": UP_SEQ_PANEL,
    "DOWN_PANEL": DOWN_PANEL,
    "DOWN_SEQ_PANEL": DOWN_SEQ_PANEL,
}
missing_panels = [name for name, value in panel_paths.items() if not value]
missing_files = [
    name for name, value in panel_paths.items() if value and not Path(value).exists()
]

if RUN_ROUTER and (missing_panels or missing_files):
    details = []
    if missing_panels:
        details.append("missing paths: " + ", ".join(missing_panels))
    if missing_files:
        details.append("paths do not exist: " + ", ".join(missing_files))
    raise RuntimeError(
        "Router panels are not ready. "
        + "; ".join(details)
        + ". Build panels or export UP_PANEL, UP_SEQ_PANEL, DOWN_PANEL, DOWN_SEQ_PANEL."
    )

inputs = pl.DataFrame(
    [
        {"name": "UP_PANEL", "value": UP_PANEL},
        {"name": "UP_SEQ_PANEL", "value": UP_SEQ_PANEL},
        {"name": "DOWN_PANEL", "value": DOWN_PANEL},
        {"name": "DOWN_SEQ_PANEL", "value": DOWN_SEQ_PANEL},
        {"name": "READINESS_RUN", "value": str(READINESS_RUN) if READINESS_RUN else ""},
        {"name": "ROUTER_RUN", "value": str(ROUTER_RUN) if ROUTER_RUN else ""},
        {"name": "REGIME_RUN", "value": str(REGIME_RUN) if REGIME_RUN else ""},
    ]
)
inputs

## 3. Execution Helpers

Heavy commands are now executable from notebook cells. If the stage flag is false, the command is printed. If the stage flag is true, output is streamed and saved to a log file.


In [ ]:
STAGE_LOGS: dict[str, Path] = {}


def print_command(command: str) -> None:
    print(command.strip() + "\n")


def stage_log_path(stage_name: str) -> Path:
    safe_stage = stage_name.replace(" ", "_").replace("/", "_")
    return LOG_ROOT / f"{NOTEBOOK_RUN_ID}_{safe_stage}.log"


def run_stage(stage_name: str, command: str, *, run: bool) -> Path | None:
    print(f"=== {stage_name} ===")
    if not run:
        print("DRY RUN: stage flag is False. Command not executed.")
        print_command(command)
        return None

    log_path = stage_log_path(stage_name)
    STAGE_LOGS[stage_name] = log_path
    print("Executing stage. Log:", log_path)
    print_command(command)

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        shell=True,
        executable="/bin/bash",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    with log_path.open("w") as log_file:
        for line in process.stdout:
            log_file.write(line)
            if STREAM_SUBPROCESS_OUTPUT:
                print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Stage {stage_name!r} failed with exit code {return_code}")
    print(f"Stage {stage_name!r} completed successfully")
    return log_path


def q(value: object) -> str:
    return shlex.quote(str(value))


def read_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text())


def read_parquet(path: Path) -> pl.DataFrame:
    if not path.exists():
        return pl.DataFrame()
    return pl.read_parquet(path)


def latest(pattern: str) -> Path | None:
    matches = sorted(
        PROJECT_ROOT.glob(pattern), key=lambda path: path.stat().st_mtime, reverse=True
    )
    return matches[0] if matches else None


def artifact_row_count(path: Path) -> int | None:
    if not path.exists():
        return None
    return int(pl.scan_parquet(path).select(pl.len()).collect().item())

## 4. Readiness

Build or reuse frozen walk-forward windows.


In [ ]:
readiness_command = f"""cd {q(PROJECT_ROOT)}
export PY={q(PY)}

"$PY" -m regression_feature_engineering.walkforward.optimize \
  --stage readiness \
  --asset {q(ASSET)} \
  --root {q(ROOT)} \
  --target-col {q(READINESS_TARGET)} \
  --config {q(CONFIG_PATH)} \
  --n-steps 400
"""
run_stage("readiness", readiness_command, run=RUN_READINESS)

## 4A. Readiness Artifact Integrity


In [ ]:
if READINESS_RUN is None:
    READINESS_RUN = latest("test_output/rpf_clean_walkforward/*_readiness_*")

if READINESS_RUN is None:
    raise RuntimeError("No readiness run found. Run Stage 1 first.")

readiness_status = read_json(READINESS_RUN / "stage_status.json", {}) or {}
readiness = read_json(READINESS_RUN / "readiness.json", {}) or {}
windows = read_parquet(READINESS_RUN / "frozen_windows.parquet")

print("READINESS_RUN:", READINESS_RUN)
print("windows:", windows.height)
print("stage_status summary:")
print(json.dumps(readiness_status.get("summary", readiness_status), indent=2)[:3000])

if windows.is_empty():
    raise RuntimeError("Readiness run has no frozen_windows.parquet rows")

cols = [
    col
    for col in ["step_idx", "pred_batch_id", "train_batch_count", "val_batch_count"]
    if col in windows.columns
]
windows.select(cols).tail(10)

## 5. Router

Run the ranked-signal router and require explicit effective artifacts.


In [ ]:
router_command = f"""cd {q(PROJECT_ROOT)}
export PY={q(PY)}
export READINESS_RUN={q(READINESS_RUN)}

"$PY" -m regression_feature_engineering.walkforward.rank_signal_router \
  --asset {q(ASSET)} \
  --root {q(ROOT)} \
  --base-run "$READINESS_RUN" \
  --up-tabular-panel-path {q(UP_PANEL)} \
  --up-sequence-panel-path {q(UP_SEQ_PANEL)} \
  --down-tabular-panel-path {q(DOWN_PANEL)} \
  --down-sequence-panel-path {q(DOWN_SEQ_PANEL)} \
  --candidate-set pruned_reliability_v1 \
  --selection-mode prequential_reliability_v1 \
  --outer-window-count 240 \
  --evaluation-block-size 20 \
  --task-type CPU \
  --thread-count 8 \
  --log-every-windows 10
"""
run_stage("rank_signal_router", router_command, run=RUN_ROUTER)

## 5A. Router Artifact Integrity


In [ ]:
if ROUTER_RUN is None:
    ROUTER_RUN = latest(
        "test_output/rpf_ranked_signal_router/*_rank_signal_router_btcusdt_8h_b"
    )

required_effective = [
    "effective_window_metrics.parquet",
    "effective_prediction_scores.parquet",
    "effective_decisions.parquet",
    "effective_block_summary.parquet",
    "effective_side_summary.json",
    "effective_artifact_source.json",
]

if ROUTER_RUN is None:
    ROUTER_RUN_IS_CLEAN = False
    source = {}
    inventory = pl.DataFrame()
    print("No router run found. Run Stage 2 first.")
else:
    missing = [name for name in required_effective if not (ROUTER_RUN / name).exists()]
    ROUTER_RUN_IS_CLEAN = not missing
    source = read_json(ROUTER_RUN / "effective_artifact_source.json", {}) or {}
    inventory = pl.DataFrame(
        [
            {
                "artifact": name,
                "exists": (ROUTER_RUN / name).exists(),
                "rows": artifact_row_count(ROUTER_RUN / name)
                if name.endswith(".parquet")
                else None,
            }
            for name in required_effective
        ]
    )
    print("ROUTER_RUN:", ROUTER_RUN)
    print("ROUTER_RUN_IS_CLEAN:", ROUTER_RUN_IS_CLEAN)
    if missing:
        print("STALE ROUTER RUN: missing effective artifacts:")
        for name in missing:
            print(" -", name)
        print("Run Stage 2 to create a fresh clean router run.")
    else:
        print("effective_artifact_source:")
        print(json.dumps(source, indent=2))

inventory

## 6. Effective Router Metrics

Strategy-performance inspection uses only effective selected artifacts.


In [ ]:
def aggregate_effective(windows: pl.DataFrame) -> pl.DataFrame:
    required = {
        "side",
        "predicted_positive_count",
        "true_positive_count",
        "false_positive_count",
        "positive_count",
        "rows",
    }
    missing = required - set(windows.columns)
    if missing:
        raise RuntimeError(
            f"effective_window_metrics missing columns: {sorted(missing)}"
        )
    return (
        windows.group_by("side")
        .agg(
            pl.len().alias("windows"),
            pl.col("predicted_positive_count").fill_null(0).sum().alias("signals"),
            pl.col("true_positive_count").fill_null(0).sum().alias("tp"),
            pl.col("false_positive_count").fill_null(0).sum().alias("fp"),
            pl.col("positive_count").fill_null(0).sum().alias("positives"),
            pl.col("rows").fill_null(0).sum().alias("rows"),
            pl.col("active_window").fill_null(False).sum().alias("active_windows")
            if "active_window" in windows.columns
            else pl.lit(None).alias("active_windows"),
        )
        .with_columns(
            (pl.col("tp") / pl.col("signals")).alias("precision"),
            (pl.col("fp") / pl.col("signals")).alias("fdr"),
            (pl.col("positives") / pl.col("rows")).alias("base_rate"),
            (
                (pl.col("tp") / pl.col("signals"))
                / (pl.col("positives") / pl.col("rows"))
            ).alias("lift"),
            (pl.col("active_windows") / pl.col("windows")).alias("active_window_rate"),
        )
        .sort("side")
    )


if not ROUTER_RUN_IS_CLEAN:
    print("No clean effective router run is available yet. Run Stage 2 first.")
    effective_windows = pl.DataFrame()
    effective_summary = pl.DataFrame()
else:
    effective_windows = read_parquet(ROUTER_RUN / "effective_window_metrics.parquet")
    effective_summary = aggregate_effective(effective_windows)

effective_summary

In [ ]:
if effective_windows.is_empty():
    print("No effective windows to display yet.")
    effective_window_view = pl.DataFrame()
else:
    view_cols = [
        col
        for col in [
            "side",
            "candidate_name",
            "selected_candidate",
            "step_idx",
            "pred_batch_id",
            "predicted_positive_count",
            "true_positive_count",
            "false_positive_count",
            "precision",
            "base_positive_rate",
            "precision_lift",
            "false_discovery_rate",
            "prediction_source",
        ]
        if col in effective_windows.columns
    ]
    effective_window_view = (
        effective_windows.select(view_cols).sort(["side", "pred_batch_id"]).tail(30)
    )

effective_window_view

## 7. Regime/Change Diagnostic

This stage uses candidate-shadow output to learn context. It does not measure strategy performance.

Required clean configuration:

- `prediction_source = candidate_shadow`
- `regime_context_mode = market_context`
- `model_mode = hmm`
- detectors:
  - `market_context_zshift_v1`
  - `market_context_recursive_cusum_v1`
  - `hmm_transition_cusum_v1`

The output must include HMM state labels, market change events, HMM transition events, and a prequential regime-gate simulation.


In [ ]:
if not ROUTER_RUN_IS_CLEAN:
    print(
        "Run Stage 2 first. Regime/change diagnostics require a clean effective router run."
    )
elif not HMM_AVAILABLE:
    print(
        "HMM is unavailable in PY environment. Install hmmlearn before running Stage 3."
    )
    print(f"{PY} -m pip install hmmlearn")
else:
    regime_command = f"""cd {q(PROJECT_ROOT)}
export PY={q(PY)}
export READINESS_RUN={q(READINESS_RUN)}

"$PY" -m regression_feature_engineering.walkforward.rank_signal_regime_diagnostic \
  --router-run {q(ROUTER_RUN)} \
  --side both \
  --candidate-names {q(ROUTER_CANDIDATE_NAMES)} \
  --prediction-source candidate_shadow \
  --regime-context-mode market_context \
  --asset {q(ASSET)} \
  --root {q(ROOT)} \
  --base-run "$READINESS_RUN" \
  --market-regime-max-features-per-family {MARKET_REGIME_MAX_FEATURES_PER_FAMILY} \
  --market-regime-lookback-batches {MARKET_REGIME_LOOKBACK_BATCHES} \
  --state-count {HMM_STATE_COUNT} \
  --hmm-selection-mode {HMM_SELECTION_MODE} \
  --hmm-state-count-choices {q(HMM_STATE_COUNT_CHOICES)} \
  --hmm-filter-mode {HMM_FILTER_MODE} \
  --pca-components {HMM_PCA_COMPONENTS} \
  --regime-min-history-windows {HMM_MIN_HISTORY_WINDOWS} \
  --regime-lookback-windows {HMM_LOOKBACK_WINDOWS} \
  --regime-refit-interval-windows {HMM_REFIT_INTERVAL_WINDOWS} \
  --model-mode hmm \
  --change-lookback-windows {CHANGE_LOOKBACK_WINDOWS} \
  --change-feature-set market_context \
  --change-detectors {q(REGIME_CHANGE_DETECTORS)} \
  --cusum-z {MARKET_ZSHIFT_THRESHOLD} \
  --recursive-cusum-k {RECURSIVE_CUSUM_K} \
  --recursive-cusum-h {RECURSIVE_CUSUM_H} \
  --cusum-standardization {CUSUM_STANDARDIZATION} \
  --cusum-target-event-rate-min {CUSUM_TARGET_EVENT_RATE_MIN} \
  --cusum-target-event-rate-max {CUSUM_TARGET_EVENT_RATE_MAX}
"""
    run_stage("regime_change_diagnostic", regime_command, run=RUN_REGIME_DIAGNOSTIC)


## 7A. Regime/Change Artifact Integrity

A clean regime run must declare prediction source, model mode, detector names, and all required artifacts. Stale runs fail here.


In [ ]:
if REGIME_RUN is None:
    REGIME_RUN = latest(
        "test_output/rpf_ranked_signal_regime_diagnostic/*_rank_signal_regime_diagnostic"
    )

required_regime_artifacts = [
    "regime_context.parquet",
    "regime_input_features.parquet",
    "market_regime_context.parquet",
    "hmm_state_metrics.parquet",
    "hmm_filter_diagnostics.parquet",
    "hmm_model_selection.parquet",
    "hmm_transition_matrix.parquet",
    "cusum_calibration.parquet",
    "change_point_events.parquet",
    "hmm_transition_events.parquet",
    "regime_signal_quality.parquet",
    "regime_target_match.parquet",
    "regime_change_target_match.parquet",
    "change_risk_target_match.parquet",
    "regime_suppression_candidates.parquet",
    "state_profile_labels.parquet",
    "regime_gate_simulation.parquet",
    "regime_gate_context_metrics.parquet",
    "regime_gate_decisions.parquet",
]
expected_detectors = set(REGIME_CHANGE_DETECTORS.split(","))

if REGIME_RUN is None:
    REGIME_RUN_IS_CLEAN = False
    regime_quality = pl.DataFrame()
    regime_quality_summary = pl.DataFrame()
    change_events = pl.DataFrame()
    hmm_transition_events = pl.DataFrame()
    state_profile_labels = pl.DataFrame()
    regime_gate_simulation = pl.DataFrame()
    hmm_filter_diagnostics = pl.DataFrame()
    hmm_model_selection = pl.DataFrame()
    hmm_transition_matrix = pl.DataFrame()
    cusum_calibration = pl.DataFrame()
    print("No regime run found. Stage 3 is optional.")
else:
    regime_cfg = read_json(REGIME_RUN / "regime_diagnostic_config.json", {}) or {}
    missing_regime_artifacts = [
        name for name in required_regime_artifacts if not (REGIME_RUN / name).exists()
    ]
    if missing_regime_artifacts:
        raise RuntimeError(
            "Regime run is stale or invalid for this notebook. Missing artifacts: "
            + ", ".join(missing_regime_artifacts)
        )
    regime_quality = read_parquet(REGIME_RUN / "regime_signal_quality.parquet")
    regime_input_features = read_parquet(REGIME_RUN / "regime_input_features.parquet")
    bad_regime_inputs = (
        regime_input_features.filter(
            (pl.col("included") == True)
            & pl.col("excluded_reason").is_not_null()
        )
        if not regime_input_features.is_empty()
        else pl.DataFrame()
    )
    if not bad_regime_inputs.is_empty():
        raise RuntimeError(
            "Regime run includes disallowed HMM/CUSUM input features"
        )
    change_events = read_parquet(REGIME_RUN / "change_point_events.parquet")
    hmm_transition_events = read_parquet(REGIME_RUN / "hmm_transition_events.parquet")
    state_profile_labels = read_parquet(REGIME_RUN / "state_profile_labels.parquet")
    regime_gate_simulation = read_parquet(REGIME_RUN / "regime_gate_simulation.parquet")
    change_risk_target_match = read_parquet(REGIME_RUN / "change_risk_target_match.parquet")
    if (
        not change_risk_target_match.is_empty()
        and "change_flag" in change_risk_target_match.columns
        and change_risk_target_match.filter(pl.col("change_flag") == "has_cusum").height > 0
    ):
        raise RuntimeError("Stale actionable change-risk artifact contains has_cusum")
    hmm_filter_diagnostics = read_parquet(REGIME_RUN / "hmm_filter_diagnostics.parquet")
    hmm_model_selection = read_parquet(REGIME_RUN / "hmm_model_selection.parquet")
    hmm_transition_matrix = read_parquet(REGIME_RUN / "hmm_transition_matrix.parquet")
    cusum_calibration = read_parquet(REGIME_RUN / "cusum_calibration.parquet")
    sources_ok = (
        not regime_quality.is_empty() and "prediction_source" in regime_quality.columns
    )
    hmm_ok = (
        regime_cfg.get("model_mode") == "hmm"
        and regime_cfg.get("hmm_selection_mode") == HMM_SELECTION_MODE
        and regime_cfg.get("hmm_filter_mode") == HMM_FILTER_MODE
    )
    detector_set = set(regime_cfg.get("change_detectors", []))
    detectors_ok = detector_set == expected_detectors
    market_ok = regime_cfg.get("regime_context_mode") == "market_context"
    cusum_ok = regime_cfg.get("cusum_standardization") == CUSUM_STANDARDIZATION
    REGIME_RUN_IS_CLEAN = bool(
        sources_ok
        and hmm_ok
        and detectors_ok
        and market_ok
        and cusum_ok
        and not missing_regime_artifacts
    )

    print("REGIME_RUN:", REGIME_RUN)
    print("REGIME_RUN_IS_CLEAN:", REGIME_RUN_IS_CLEAN)
    print("prediction_source:", regime_cfg.get("prediction_source"))
    print("model_mode:", regime_cfg.get("model_mode"))
    print("hmm_selection_mode:", regime_cfg.get("hmm_selection_mode"))
    print("hmm_filter_mode:", regime_cfg.get("hmm_filter_mode"))
    print("cusum_standardization:", regime_cfg.get("cusum_standardization"))
    print("change_detectors:", regime_cfg.get("change_detectors"))
    print("regime_context_mode:", regime_cfg.get("regime_context_mode"))
    print("market change events:", change_events.height)
    print("HMM transition events:", hmm_transition_events.height)
    print("state profile labels:", state_profile_labels.height)
    print("gate simulation rows:", regime_gate_simulation.height)
    print("regime input features:", regime_input_features.height)
    print("HMM filter diagnostics:", hmm_filter_diagnostics.height)
    print("HMM model-selection rows:", hmm_model_selection.height)
    print("HMM transition-matrix rows:", hmm_transition_matrix.height)
    print("CUSUM calibration rows:", cusum_calibration.height)
    if missing_regime_artifacts:
        print("missing artifacts:", missing_regime_artifacts)
    if not sources_ok:
        print(
            "STALE/INVALID: regime_signal_quality.parquet missing prediction_source or empty"
        )
    if not detectors_ok:
        print("STALE/INVALID: detector set does not match notebook clean config")
    if not hmm_ok:
        print("STALE/INVALID: HMM mode/selection/filter does not match notebook clean config")
    if not cusum_ok:
        print("STALE/INVALID: CUSUM standardization does not match notebook clean config")
    if REGIME_RUN is not None and not REGIME_RUN_IS_CLEAN:
        raise RuntimeError(
            "Regime run is stale or invalid for this notebook. Rerun Stage 7."
        )
    print(
        "prediction sources:",
        regime_quality.select("prediction_source").unique().to_series().to_list(),
    )
    regime_quality_summary = aggregate_effective(regime_quality)

regime_quality_summary


## 8. HMM State Interpretation

Checks whether HMM produced enough non-degenerate market states and whether those states are stable enough to inspect.


In [ ]:
def metric_status(ok: bool) -> str:
    return "pass" if ok else "fail"


if not REGIME_RUN_IS_CLEAN:
    print("No clean HMM/CUSUM run available. Run Stage 3 first.")
    hmm_state_windows = pl.DataFrame()
    hmm_health = pl.DataFrame()
else:
    regime_context = read_parquet(REGIME_RUN / "regime_context.parquet")
    market_context = read_parquet(REGIME_RUN / "market_regime_context.parquet")
    state_metrics = read_parquet(REGIME_RUN / "hmm_state_metrics.parquet")
    hmm_filter_diagnostics = read_parquet(REGIME_RUN / "hmm_filter_diagnostics.parquet")
    clean_state_rows = (
        regime_context.filter(pl.col("regime_status") == "ok")
        if "regime_status" in regime_context.columns
        else pl.DataFrame()
    )
    unique_cols = [
        col
        for col in [
            "source_idx",
            "pred_batch_id",
            "regime_state",
            "regime_state_key",
            "regime_posterior_max",
            "regime_entropy",
            "regime_changed_from_previous",
            "regime_selected_state_count",
            "regime_filter_mode",
            "regime_current_log_likelihood",
            "regime_hmm_degenerate_flag",
            "regime_max_self_transition_probability",
            "regime_max_expected_state_duration",
        ]
        if col in clean_state_rows.columns
    ]
    hmm_state_windows = (
        clean_state_rows.select(unique_cols)
        .unique(subset=["source_idx", "pred_batch_id"], keep="first")
        .sort(["source_idx", "pred_batch_id"])
        if unique_cols
        else pl.DataFrame()
    )
    total_market_windows = market_context.height
    assigned_windows = hmm_state_windows.height
    active_states = (
        hmm_state_windows.select("regime_state_key").unique().height
        if "regime_state_key" in hmm_state_windows.columns
        and not hmm_state_windows.is_empty()
        else 0
    )
    max_state_share = None
    posterior_mean = None
    entropy_mean = None
    if not hmm_state_windows.is_empty():
        state_counts = hmm_state_windows.group_by("regime_state_key").agg(
            pl.len().alias("windows")
        )
        max_state_share = float(state_counts["windows"].max() / assigned_windows)
        posterior_mean = (
            float(hmm_state_windows["regime_posterior_max"].mean())
            if "regime_posterior_max" in hmm_state_windows.columns
            else None
        )
        entropy_mean = (
            float(hmm_state_windows["regime_entropy"].mean())
            if "regime_entropy" in hmm_state_windows.columns
            else None
        )
    degenerate_windows = (
        int(hmm_state_windows["regime_hmm_degenerate_flag"].fill_null(False).sum())
        if "regime_hmm_degenerate_flag" in hmm_state_windows.columns
        and not hmm_state_windows.is_empty()
        else 0
    )
    selected_counts = (
        hmm_filter_diagnostics.select("regime_selected_state_count").drop_nulls().unique().height
        if not hmm_filter_diagnostics.is_empty()
        and "regime_selected_state_count" in hmm_filter_diagnostics.columns
        else 0
    )
    state_count_cfg = int(regime_cfg.get("state_count", 3) or 3)
    max_entropy = float(__import__("math").log(max(state_count_cfg, 2)))
    hmm_health = pl.DataFrame(
        [
            {
                "check": "market_context_rows",
                "value": total_market_windows,
                "threshold": ">= regime_min_history_windows + 20",
                "status": metric_status(
                    total_market_windows
                    >= int(regime_cfg.get("regime_min_history_windows", 80) or 80) + 20
                ),
            },
            {
                "check": "assigned_hmm_windows",
                "value": assigned_windows,
                "threshold": ">= 20",
                "status": metric_status(assigned_windows >= 20),
            },
            {
                "check": "bic_selected_state_count_variants",
                "value": selected_counts,
                "threshold": ">= 1",
                "status": metric_status(selected_counts >= 1),
            },
            {
                "check": "active_state_count",
                "value": active_states,
                "threshold": ">= 2",
                "status": metric_status(active_states >= 2),
            },
            {
                "check": "max_state_share",
                "value": max_state_share,
                "threshold": "<= 0.80",
                "status": metric_status(
                    max_state_share is not None and max_state_share <= 0.80
                ),
            },
            {
                "check": "posterior_max_mean",
                "value": posterior_mean,
                "threshold": ">= 0.60",
                "status": metric_status(
                    posterior_mean is not None and posterior_mean >= 0.60
                ),
            },
            {
                "check": "entropy_mean",
                "value": entropy_mean,
                "threshold": f"<= 0.85 * log({state_count_cfg})",
                "status": metric_status(
                    entropy_mean is not None and entropy_mean <= 0.85 * max_entropy
                ),
            },
            {
                "check": "hmm_degenerate_windows",
                "value": degenerate_windows,
                "threshold": "== 0",
                "status": metric_status(degenerate_windows == 0),
            },
        ]
    )

hmm_health


In [ ]:
if hmm_state_windows.is_empty():
    hmm_state_coverage = pl.DataFrame()
else:
    agg_exprs = [pl.len().alias("windows")]
    if "regime_posterior_max" in hmm_state_windows.columns:
        agg_exprs.append(
            pl.col("regime_posterior_max").mean().alias("posterior_max_mean")
        )
        agg_exprs.append(
            pl.col("regime_posterior_max").min().alias("posterior_max_min")
        )
    if "regime_entropy" in hmm_state_windows.columns:
        agg_exprs.append(pl.col("regime_entropy").mean().alias("entropy_mean"))
    if "regime_changed_from_previous" in hmm_state_windows.columns:
        agg_exprs.append(
            pl.col("regime_changed_from_previous")
            .fill_null(False)
            .sum()
            .alias("state_change_count")
        )
    hmm_state_coverage = (
        hmm_state_windows.group_by("regime_state_key")
        .agg(*agg_exprs)
        .with_columns(
            (pl.col("windows") / pl.col("windows").sum()).alias("window_share")
        )
        .sort("regime_state_key")
    )

hmm_state_coverage

## 8A. HMM State Transition Table

Check whether HMM states transition in a plausible way instead of randomly jumping every window.


In [ ]:
if hmm_state_windows.is_empty():
    hmm_state_transition_table = pl.DataFrame()
else:
    hmm_state_transition_table = (
        hmm_state_windows.sort(["source_idx", "pred_batch_id"])
        .with_columns(
            pl.col("regime_state_key").shift(1).over("source_idx").alias("previous_regime_state_key")
        )
        .filter(pl.col("previous_regime_state_key").is_not_null())
        .group_by(["previous_regime_state_key", "regime_state_key"])
        .agg(pl.len().alias("transitions"))
        .with_columns(
            (pl.col("transitions") / pl.col("transitions").sum().over("previous_regime_state_key")).alias("transition_share")
        )
        .sort(["previous_regime_state_key", "regime_state_key"])
    )

hmm_state_transition_table


## 8B. HMM State Feature Profiles

Show which market-context features separate states and the automatic state-profile labels produced by the diagnostic command.


In [ ]:
def top_state_separating_market_features(
    frame: pl.DataFrame, *, top_n: int = 12
) -> tuple[list[str], pl.DataFrame]:
    if frame.is_empty() or "regime_state_key" not in frame.columns:
        return [], pl.DataFrame()
    market_cols = [
        col
        for col in frame.columns
        if col.startswith("market_")
        and "context_" not in col
        and "feature_count" not in col
        and "row_count" not in col
        and "batch_count" not in col
        and "finite_rate" not in col
        and (
            col.endswith("_last_z")
            or col.endswith("_lookback_mean")
            or col.endswith("_last")
        )
    ]
    if not market_cols:
        return [], pl.DataFrame()
    means = frame.group_by("regime_state_key").agg(
        *[
            pl.col(col).cast(pl.Float64, strict=False).mean().alias(col)
            for col in market_cols
        ]
    )
    spreads = []
    for col in market_cols:
        values = means[col].drop_nulls()
        if values.len() < 2:
            continue
        spread = float(values.max() - values.min())
        if spread == spread:
            spreads.append({"feature": col, "state_mean_spread": spread})
    spread_df = pl.DataFrame(spreads).sort("state_mean_spread", descending=True)
    top_cols = (
        spread_df.head(top_n)["feature"].to_list() if not spread_df.is_empty() else []
    )
    profile = (
        means.select(["regime_state_key", *top_cols]).sort("regime_state_key")
        if top_cols
        else pl.DataFrame()
    )
    return top_cols, profile


if not REGIME_RUN_IS_CLEAN or hmm_state_windows.is_empty():
    top_hmm_market_features = []
    hmm_market_profile = pl.DataFrame()
else:
    # Use one row per prediction window to avoid duplicating market context by candidate.
    unique_market_state = (
        regime_context.filter(pl.col("regime_status") == "ok")
        .unique(subset=["source_idx", "pred_batch_id"], keep="first")
        .sort(["source_idx", "pred_batch_id"])
    )
    top_hmm_market_features, hmm_market_profile = top_state_separating_market_features(
        unique_market_state, top_n=12
    )

print("top market features:")
for feature in top_hmm_market_features:
    print("-", feature)

hmm_market_profile

## 8C. State Profile Labels

These labels are deterministic descriptions from market feature profiles. They are not trading decisions.


In [ ]:
state_profile_labels if "state_profile_labels" in globals() else pl.DataFrame()


## 9. CUSUM And HMM-Transition Checks

Check market z-shifts, recursive market CUSUM, HMM-transition CUSUM, and whether alarms align with HMM state changes or candidate quality changes.


In [ ]:
def detector_window_count(frame: pl.DataFrame, column: str) -> int:
    if frame.is_empty() or column not in frame.columns:
        return 0
    return (
        frame.filter(pl.col(column).fill_null(False))
        .select(["source_idx", "pred_batch_id"])
        .unique()
        .height
    )


def detector_name_contract_check(frame: pl.DataFrame, artifact_name: str) -> None:
    if frame.is_empty() or "detector_name" not in frame.columns:
        return
    bad = (
        frame.filter(pl.col("detector_name").str.contains(",", literal=True))
        .select("detector_name")
        .unique()
        .sort("detector_name")
    )
    if not bad.is_empty():
        raise RuntimeError(
            f"{artifact_name} has non-explicit detector_name values. "
            "Rerun the regime diagnostic with the clean detector writer. Bad names: "
            + ", ".join(bad["detector_name"].to_list())
        )


if not REGIME_RUN_IS_CLEAN:
    cusum_health = pl.DataFrame()
    cusum_feature_counts = pl.DataFrame()
else:
    detector_name_contract_check(change_events, "change_point_events.parquet")
    detector_name_contract_check(hmm_transition_events, "hmm_transition_events.parquet")

    total_market_windows = market_context.height if "market_context" in globals() else 0
    calibration = read_parquet(REGIME_RUN / "cusum_calibration.parquet")
    marker_counts = pl.DataFrame(
        [
            {
                "detector_name": "hmm_state_change_marker",
                "event_windows": detector_window_count(hmm_transition_events, "hmm_state_change_alarm"),
                "total_windows": total_market_windows,
                "event_window_rate": (
                    detector_window_count(hmm_transition_events, "hmm_state_change_alarm")
                    / total_market_windows
                    if total_market_windows
                    else None
                ),
                "calibration_status": "marker",
            }
        ]
    )
    cusum_health = (
        pl.concat([calibration, marker_counts], how="diagonal_relaxed")
        if not calibration.is_empty()
        else marker_counts
    )
    feature_frames = []
    if not change_events.is_empty() and "feature" in change_events.columns:
        feature_frames.append(
            change_events.group_by(["detector_name", "feature"])
            .agg(
                pl.len().alias("events"),
                pl.col("zscore").abs().mean().alias("mean_abs_zscore"),
            )
        )
    if not hmm_transition_events.is_empty() and "feature" in hmm_transition_events.columns:
        feature_frames.append(
            hmm_transition_events.group_by(["detector_name", "feature"])
            .agg(
                pl.len().alias("events"),
                pl.col("zscore").abs().mean().alias("mean_abs_zscore"),
            )
        )
    cusum_feature_counts = (
        pl.concat(feature_frames, how="diagonal_relaxed")
        .sort("events", descending=True)
        .head(30)
        if feature_frames
        else pl.DataFrame()
    )

cusum_health


In [ ]:
cusum_feature_counts

In [ ]:
if not REGIME_RUN_IS_CLEAN:
    change_candidate_separation = pl.DataFrame()
else:
    change_match = read_parquet(REGIME_RUN / "change_risk_target_match.parquet")
    wanted_flags = [
        "has_market_context_zshift_v1",
        "has_market_context_recursive_cusum_v1",
        "has_hmm_transition_cusum_v1",
        "has_hmm_state_change",
    ]
    change_candidate_separation = (
        change_match.filter(pl.col("change_flag").is_in(wanted_flags))
        .select(
            [
                "side",
                "candidate_name",
                "change_flag",
                "change_flag_value",
                "signals",
                "precision",
                "base_rate",
                "precision_lift",
                "false_discovery_rate",
                "target_match_status",
            ]
        )
        .sort(["side", "candidate_name", "change_flag", "change_flag_value"])
        if not change_match.is_empty() and "change_flag" in change_match.columns
        else pl.DataFrame()
    )

change_candidate_separation


## 9A. HMM/CUSUM Alignment

Check whether market CUSUM or HMM-transition CUSUM fires on or before HMM state-change windows.


In [ ]:
if not REGIME_RUN_IS_CLEAN or hmm_state_windows.is_empty():
    hmm_cusum_alignment = pl.DataFrame()
else:
    change_flags = read_parquet(REGIME_RUN / "regime_change_target_match.parquet")
    transition_windows = (
        hmm_state_windows.filter(pl.col("regime_changed_from_previous") == True)  # noqa: E712
        .select(["source_idx", "pred_batch_id"])
        .unique()
    )
    market_alarm_windows = (
        change_events.select(["source_idx", "pred_batch_id"]).unique()
        if not change_events.is_empty()
        else pl.DataFrame({"source_idx": [], "pred_batch_id": []})
    )
    hmm_transition_cusum_windows = (
        hmm_transition_events.filter(pl.col("hmm_transition_cusum_v1_alarm").fill_null(False))
        .select(["source_idx", "pred_batch_id"])
        .unique()
        if not hmm_transition_events.is_empty()
        and "hmm_transition_cusum_v1_alarm" in hmm_transition_events.columns
        else pl.DataFrame({"source_idx": [], "pred_batch_id": []})
    )
    hmm_state_marker_windows = (
        hmm_transition_events.filter(pl.col("hmm_state_change_alarm").fill_null(False))
        .select(["source_idx", "pred_batch_id"])
        .unique()
        if not hmm_transition_events.is_empty()
        and "hmm_state_change_alarm" in hmm_transition_events.columns
        else pl.DataFrame({"source_idx": [], "pred_batch_id": []})
    )
    hmm_cusum_alignment = pl.DataFrame(
        [
            {
                "item": "hmm_state_change_windows",
                "value": transition_windows.height,
            },
            {
                "item": "state_changes_with_market_change_alarm",
                "value": transition_windows.join(
                    market_alarm_windows, on=["source_idx", "pred_batch_id"], how="inner"
                ).height,
            },
            {
                "item": "state_changes_with_hmm_transition_cusum_alarm",
                "value": transition_windows.join(
                    hmm_transition_cusum_windows,
                    on=["source_idx", "pred_batch_id"],
                    how="inner",
                ).height,
            },
            {
                "item": "state_changes_with_hmm_state_change_marker",
                "value": transition_windows.join(
                    hmm_state_marker_windows,
                    on=["source_idx", "pred_batch_id"],
                    how="inner",
                ).height,
            },
        ]
    )

hmm_cusum_alignment


## 10. Regime-Aware Gate Simulation

Replay a conservative prequential regime gate from prior matured windows only. This does not modify router decisions.


In [ ]:
if not REGIME_RUN_IS_CLEAN:
    regime_gate_summary = pl.DataFrame()
    regime_actionability = pl.DataFrame()
else:
    gate = read_parquet(REGIME_RUN / "regime_gate_simulation.parquet")
    target_match = read_parquet(REGIME_RUN / "regime_target_match.parquet")
    suppression_candidates = read_parquet(
        REGIME_RUN / "regime_suppression_candidates.parquet"
    )
    favorable = (
        target_match.filter(pl.col("target_match_status") == "favorable")
        .select(
            [
                "side",
                "candidate_name",
                "regime_state_key",
                "signals",
                "precision",
                "base_rate",
                "precision_lift",
                "false_discovery_rate",
                "target_match_score",
            ]
        )
        .sort("target_match_score", descending=True)
        if not target_match.is_empty()
        else pl.DataFrame()
    )
    avoid = (
        suppression_candidates.select(
            [
                col
                for col in [
                    "side",
                    "candidate_name",
                    "suppression_context",
                    "signals",
                    "precision",
                    "base_rate",
                    "precision_lift",
                    "false_discovery_rate",
                    "suppression_score",
                ]
                if col in suppression_candidates.columns
            ]
        ).sort("suppression_score", descending=True)
        if not suppression_candidates.is_empty()
        else pl.DataFrame()
    )
    regime_gate_summary = (
        gate.group_by(["side", "candidate_name"])
        .agg(
            pl.len().alias("windows"),
            pl.col("signals_before").sum().alias("signals_before"),
            pl.col("true_positives_before").sum().alias("tp_before"),
            pl.col("false_positives_before").sum().alias("fp_before"),
            pl.col("signals_after").sum().alias("signals_after"),
            pl.col("true_positives_after").sum().alias("tp_after"),
            pl.col("false_positives_after").sum().alias("fp_after"),
            (pl.col("gate_action") == "allow").sum().alias("allowed_windows"),
        )
        .with_columns(
            (pl.col("tp_before") / pl.col("signals_before")).alias("precision_before"),
            (pl.col("tp_after") / pl.col("signals_after")).alias("precision_after"),
            (pl.col("fp_before") / pl.col("signals_before")).alias("fdr_before"),
            (pl.col("fp_after") / pl.col("signals_after")).alias("fdr_after"),
            (pl.col("allowed_windows") / pl.col("windows")).alias("allowed_window_rate"),
            (pl.col("signals_after") / (pl.col("windows") / 21.0)).alias(
                "estimated_signals_per_week"
            ),
        )
        .sort(["side", "candidate_name"])
        if not gate.is_empty()
        else pl.DataFrame()
    )
    rows = [
        {
            "item": "hmm_has_actionable_favorable_contexts",
            "value": favorable.height,
            "status": metric_status(favorable.height > 0),
        },
        {
            "item": "cusum_or_regime_has_avoid_contexts",
            "value": avoid.height,
            "status": metric_status(avoid.height > 0),
        },
        {
            "item": "regime_gate_simulation_has_after_signals",
            "value": int(regime_gate_summary["signals_after"].sum())
            if not regime_gate_summary.is_empty()
            else 0,
            "status": metric_status(
                not regime_gate_summary.is_empty()
                and int(regime_gate_summary["signals_after"].sum()) > 0
            ),
        },
    ]
    regime_actionability = pl.DataFrame(rows)

regime_actionability


In [ ]:
print("Regime gate simulation summary")
regime_gate_summary if "regime_gate_summary" in globals() else pl.DataFrame()


In [ ]:
print("Favorable HMM contexts")
favorable if "favorable" in globals() else pl.DataFrame()


In [ ]:
print("Avoid/suppression candidates")
avoid if "avoid" in globals() else pl.DataFrame()


## 11. Validation

Run this after editing code or notebook cells.


In [ ]:
validation_command = f"""cd {q(PROJECT_ROOT)}
python -m pytest tests/test_rpf*.py -q
python -m py_compile $(find regression_feature_engineering -name '*.py' -print)
python -m ruff check notebooks/rpf_walkforward_control.ipynb
git diff --check
"""
run_stage("validation", validation_command, run=RUN_VALIDATION)

## 12. Worktree/Artifact Check

Use before commit. Local `test_output` artifacts should usually stay untracked and local.


In [ ]:
worktree_command = f"""cd {q(PROJECT_ROOT)}
git status --short
git diff --stat
"""
run_stage("worktree_check", worktree_command, run=RUN_WORKTREE_CHECK)

## 13. Decision Checklist

Do not continue modeling unless all are true:

1. Readiness has frozen windows and label safety.
2. Router run has explicit `effective_*` artifacts.
3. Effective metrics are inspected for both UP and DOWN.
4. Strategy-performance conclusions use effective selected output only.
5. HMM regime diagnostic has `model_mode=hmm` and `regime_context_mode=market_context`.
6. Detector names are explicit: `market_context_zshift_v1`, `market_context_recursive_cusum_v1`, and `hmm_transition_cusum_v1`.
7. HMM diagnostics pass coverage/stability checks and show separated market feature profiles.
8. CUSUM/HMM-transition diagnostics have sane event rates and separate at least one candidate context.
9. Regime-gate simulation improves precision/lift/FDR without collapsing practical signal frequency.
10. No strategy-performance conclusion comes from shadow candidate artifacts or stale regime artifacts.

If any item fails, fix the implementation or stop the branch.
